# GP1 — Netflix Shows: Exploratory Data Analysis (EDA)
**Фокус:** очистка данных, EDA, создание дополнительных метрик успеха (SuccessIndex, rating_density, Balance, RA_Index) и интерпретация результатов.

Этот ноутбук — аккуратно оформленная версия моего учебного проекта.

## 1) Импорт библиотек

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 5)
sns.set(style='whitegrid')

## 2) Загрузка данных

Ожидается файл **NetflixShows.xlsx** рядом с ноутбуком.

In [ ]:
data = pd.read_excel('NetflixShows.xlsx')
orig_data = data.copy()
data.head()

## 3) Первичный обзор

In [ ]:
print('shape:', data.shape)
print('\ncolumns:', list(data.columns))
print('\ndtypes:\n', data.dtypes)

## 4) Приведение заголовков и строк к единому виду

In [ ]:
data.columns = (
    data.columns
    .str.replace(' ', '_')
    .str.lower()
    .str.strip()
)

# очистка пробелов в строковых колонках (как в исходном проекте)
for col in ['title', 'rating', 'ratinglevel']:
    if col in data.columns:
        data[col] = data[col].astype('string').str.strip()

data.columns

## 5) Дубликаты

Считаю два типа:
- **полные дубликаты строк**
- дубликаты по ключу **(title, release_year)** (один и тот же фильм с разными значениями).

In [ ]:
full_dupl_count = data.duplicated().sum()
data = data.drop_duplicates(keep='first')

count_title_release_dupl = data.duplicated(subset=['title', 'release_year']).sum()

print('shape after drop full duplicates:', data.shape)
print('full duplicates removed:', full_dupl_count)
print('duplicates by (title, release_year):', count_title_release_dupl)

Посмотрим на дубликаты по (title, release_year), если они есть.

In [ ]:
mask = data.duplicated(subset=['title', 'release_year'], keep=False)
title_release_dupl = data[mask]
print(title_release_dupl)

Оставляем запись с меньшим числом пропусков (Nan_count).

In [ ]:
data['Nan_count'] = data.isna().sum(axis=1)
data = data.sort_values(by='Nan_count')
data = data.drop_duplicates(subset=['title', 'release_year'], keep='first')
data = data.drop(columns='Nan_count')
print('shape:', data.shape)

## 6) Рейтинговые группы и пропуски

Заменяем `UR/NR` (нет рейтинга) на `NaN`, затем анализируем пропуски.

In [ ]:
rating_groups = (data['rating'].unique()).size
print('Количество уникальных rating-групп:', rating_groups)

data['rating'] = data['rating'].replace(['UR', 'NR'], np.nan)

def missing_values_table(df):
    mis_val = df.isnull().sum()
    mis_val_percent = 100 * df.isnull().sum() / len(df)
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
    mis_val_table_ren_columns = mis_val_table.rename(columns={0: 'Missing Values', 1: '% of Total Values'})
    mis_val_table_ren_columns = mis_val_table_ren_columns[mis_val_table_ren_columns.iloc[:,1] != 0].sort_values('% of Total Values', ascending=False).round(2)
    return mis_val_table_ren_columns

missing_values_table(data)

По проекту: `ratinglevel` и `rating` имеют небольшой процент пропусков — удаляем строки.

In [ ]:
data = data.dropna(subset=['ratinglevel'])
data = data.dropna(subset=['rating'])
print('shape:', data.shape)

### Заполнение пропусков в целевой переменной `user_rating_score`

В исходной работе пропуски (~высокая доля) заполнялись **медианой внутри когорты `ratingdescription`** + небольшое случайное отклонение (noise), чтобы не «сплющивать» распределение.

In [ ]:
median_by_ratingdescription = (
    data.groupby('ratingdescription')['user_rating_score']
        .median()
        .reset_index(name='median_rating_score')
)

# объединяем с таблицей медиан
data = data.merge(median_by_ratingdescription, on='ratingdescription', how='left')

# заполняем пропуски медианой + случайным отклонением
mask = data['user_rating_score'].isna()
n_missing = mask.sum()
np.random.seed(42)

data.loc[mask, 'user_rating_score'] = (
    data.loc[mask, 'median_rating_score'] + np.random.normal(0, 1, n_missing)
)

# удаляем вспомогательный столбец
data = data.drop(columns=['median_rating_score'])

data['user_rating_score'].isna().sum()

## 7) EDA: распределение оценок пользователей

In [ ]:
sns.displot(data['user_rating_score'], kde=True, bins=25, height=6, aspect=2)
plt.xticks(np.arange(56, 100, 2))
plt.xlabel('Оценка')
plt.ylabel('Количество')
plt.title('Гистограмма оценок')
plt.show()

unique_user_rating_score = data['user_rating_score'].nunique()
print('Количество уникальных оценок:', unique_user_rating_score)

In [ ]:
sns.boxplot(y=data['user_rating_score'])
plt.title('Boxplot целевой переменной')
plt.show()

print('Средняя оценка:', data['user_rating_score'].mean())
print('Стандартное отклонение:', data['user_rating_score'].std())
print('Медиана:', data['user_rating_score'].median())
print('Мин:', data['user_rating_score'].min())
print('Макс:', data['user_rating_score'].max())

## 8) Новый набор признаков (Feature Engineering)

В проекте я добавил показатели, которые помогают анализировать не только «качество» (score), но и «масштаб аудитории» (size), а также нормировать популярность на возраст контента.

In [ ]:
# возраст контента (в исходном проекте использовался 2025 год)
data['content_age'] = 2025 - data['release_year']

# распределение по декадам
data['decade'] = (data['release_year'] // 10 * 10).astype(int).astype(str) + 's'

# SuccessIndex
data['SuccessIndex'] = data['user_rating_score']/100 * np.log(data['user_rating_size'] + 1)

# rating_density
data['rating_density'] = data['user_rating_size']/(data['content_age'] + 1)

# Balance: Z-score внутри rating-категории
_data_mean = data.groupby('rating')['user_rating_score'].transform('mean')
_data_std = data.groupby('rating')['user_rating_score'].transform('std')
data['Balance'] = (data['user_rating_score'] - _data_mean) / _data_std

# RA_Index
data['RA_Index'] = (data['user_rating_score'] / 100) * (np.log(data['user_rating_size'] + 1) / np.log(data['content_age'] + 2))

data[['title','release_year','rating','user_rating_score','user_rating_size','SuccessIndex','rating_density','Balance','RA_Index']].head()

## 9) Корреляции (heatmap)

Важно: некоторые корреляции «высокие», потому что метрики построены друг на друге (например, RA_Index включает log(size) и нормировку на возраст).

In [ ]:
data_for_heatmap = data[['user_rating_score','user_rating_size','ratingdescription','release_year','RA_Index','rating_density','SuccessIndex','Balance']]

plt.figure(figsize=(12, 8))
sns.heatmap(data_for_heatmap.corr(), cbar=True, annot=True, fmt='.2f')
plt.title('Correlation heatmap')
plt.show()

## 10) 3D-анализ по группам (rating × decade)

В исходном проекте мы строили интерактивные 3D-графики для разных метрик. Ниже — функции из проекта.

In [ ]:
try:
    import plotly.express as px

    def plot_interactive_3d(df, metric, agg='mean', rating_col='rating', decade_col='decade'):
        grp = (df
               .pivot_table(values=metric, index=[rating_col, decade_col], aggfunc=agg)
               .reset_index()
               .sort_values([decade_col, rating_col]))

        # перевод десятилетия в число для оси
        grp['_decade_num'] = grp[decade_col].astype(str).str.extract(r'(\d+)').astype(float)

        fig = px.scatter_3d(
            grp,
            x='_decade_num',
            y=rating_col,
            z=metric,
            color=rating_col,
            title=f"{metric}: {agg} by {rating_col} × {decade_col}",
            labels={'_decade_num': 'decade'}
        )
        fig.show()

    def plot_interactive_3d_raw(df, metric, rating_col='rating', decade_col='decade', sample=None):
        _df = df.sample(sample, random_state=42) if (sample and len(df) > sample) else df.copy()
        _df['_decade_num'] = _df[decade_col].astype(str).str.extract(r'(\d+)').astype(float)

        fig = px.scatter_3d(
            _df,
            x='_decade_num',
            y=rating_col,
            z=metric,
            color=rating_col,
            title=f"{metric}: raw points by {rating_col} × {decade_col}",
            labels={'_decade_num': 'decade'}
        )
        fig.show()

except Exception as e:
    print('Plotly блок пропущен:', e)

## 11) Срез по SuccessIndex: популярные шоу

Идея блока: выделить в каждой декаде шоу с наибольшим SuccessIndex и посмотреть распределение рейтинговых категорий.

In [ ]:
columns = 'title rating ratinglevel ratingdescription release_year user_rating_score user_rating_size content_age decade SuccessIndex rating_density Balance RA_Index'.split()

data_success_decade = (data.groupby('decade')['SuccessIndex']).idxmax()

popular = data.loc[data_success_decade, columns].sort_values('decade')
popular

Распределение `rating` для популярных шоу по декадам (pie charts) — как в исходном проекте.

In [ ]:
popular_final = popular.copy()

rating_counts_popular = popular_final.groupby(['decade', 'rating']).size().reset_index(name='count')
decades_popular = popular_final['decade'].unique()

fig, axes = plt.subplots(1, len(decades_popular), figsize=(4 * len(decades_popular), 5))
if len(decades_popular) == 1:
    axes = [axes]

for i, dec in enumerate(decades_popular):
    sec = rating_counts_popular[rating_counts_popular['decade'] == dec]
    axes[i].pie(sec['count'], labels=sec['rating'], autopct='%d')
    axes[i].set_title(dec)

plt.suptitle('Rating distribution for Top SuccessIndex shows by decade')
plt.show()

## 12) Итоговые выводы

- Я удалил полные дубликаты (в исходных данных их было много) и аккуратно обработал редкий дубль по `(title, release_year)`.
- Небольшие пропуски в `rating`/`ratinglevel` удалены, а пропуски в `user_rating_score` заполнены медианой по `ratingdescription` + noise (чтобы сохранить форму распределения).
- Для анализа «успеха» контента я ввёл метрики, учитывающие и качество, и популярность, а также нормировку на возраст контента: `SuccessIndex`, `rating_density`, `Balance`, `RA_Index`.
- Корреляции интерпретировал осторожно: часть метрик зависит друг от друга по формуле.

**Как бы применил это в продуктовой аналитике:** выделение успешных сегментов контента (по decade×rating), проверка гипотез о факторах успеха и постановка A/B-экспериментов вокруг рекомендаций/витрин/каталога.